# 1.05 — Loops — solutions

Lesson: [`lessons/1.05_loops.md`](../lessons/1.05_loops.md)

Before running any loop, fill a trace table on paper for the smallest test case.

> **Reference solutions.** Open only after you have solved the problem yourself or climbed the hint ladder. Compare, don't copy: then re-solve from blank in a few days.

In [ ]:
# ▶ Run this cell first. It defines check() and expect(), which test your answers,
#   and DATA, the path to the course's shared data folder.
import copy, math
from pathlib import Path

DATA = next((p / "data" for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "README.md").exists()), Path("data"))

def _same(a, b):
    if hasattr(b, "shape") and hasattr(b, "dtype"):          # numpy arrays
        import numpy as np
        try:
            return np.shape(a) == np.shape(b) and np.allclose(a, b, rtol=1e-5, atol=1e-8)
        except TypeError:
            return False
    if isinstance(b, float) and isinstance(a, (int, float)) and not isinstance(a, bool):
        return math.isclose(a, b, rel_tol=1e-6, abs_tol=1e-9)
    if isinstance(b, (list, tuple)):
        return type(a) is type(b) and len(a) == len(b) and all(_same(x, y) for x, y in zip(a, b))
    if isinstance(b, dict):
        return isinstance(a, dict) and a.keys() == b.keys() and all(_same(a[k], b[k]) for k in b)
    return a == b

def check(fn, cases):
    """Run fn on each (args, expected) pair and report. args is a tuple of arguments."""
    ok = 0
    for args, expected in cases:
        if not isinstance(args, tuple):
            args = (args,)
        shown = ", ".join(repr(a) for a in args)
        if len(shown) > 80:
            shown = shown[:77] + "..."
        try:
            got = fn(*copy.deepcopy(args))
        except Exception as e:
            print(f"  ✗ {fn.__name__}({shown}) raised {type(e).__name__}: {e}")
            continue
        if _same(got, expected):
            ok += 1
        else:
            print(f"  ✗ {fn.__name__}({shown}) returned {got!r}, expected {expected!r}")
    status = "✅ all passed" if ok == len(cases) else "keep going"
    print(f"{fn.__name__}: {ok}/{len(cases)} passed — {status}")

def expect(description, condition):
    """For checks that aren't a simple function call: expect('mean is ~0', abs(m) < 1e-6)."""
    try:
        passed = bool(condition() if callable(condition) else condition)
    except Exception as e:
        print(f"  ✗ {description} — raised {type(e).__name__}: {e}")
        return
    print(("  ✅ " if passed else "  ✗ ") + description + ("" if passed else " — keep going"))


## Problem 1 — sum to n

`sum_to(n)` returns 1 + 2 + … + n using a loop (not the formula). Then check your answer against n(n+1)/2 in a scratch cell.

In [ ]:
def sum_to(n):
    total = 0
    for i in range(1, n + 1):
        total += i
    return total

In [ ]:
check(sum_to, [((1,), 1), ((10,), 55), ((100,), 5050), ((0,), 0)])

## Problem 2 — count vowels

`count_vowels(text)` counts a, e, i, o, u in any case.

In [ ]:
def count_vowels(text):
    count = 0
    for ch in text.lower():
        if ch in "aeiou":
            count += 1
    return count

In [ ]:
check(count_vowels, [(("Data Science",), 5), (("rhythm",), 0), (("AEIOU",), 5), (("",), 0)])

## Problem 3 — FizzBuzz

`fizzbuzz(n)` returns a **list** of strings for 1..n: `"Fizz"` for multiples of 3, `"Buzz"` for multiples of 5, `"FizzBuzz"` for both, otherwise the number as a string.

In [ ]:
def fizzbuzz(n):
    out = []
    for i in range(1, n + 1):
        if i % 15 == 0:
            out.append("FizzBuzz")
        elif i % 3 == 0:
            out.append("Fizz")
        elif i % 5 == 0:
            out.append("Buzz")
        else:
            out.append(str(i))
    return out

In [ ]:
check(fizzbuzz, [((5,), ["1", "2", "Fizz", "4", "Buzz"]), ((15,), ["1", "2", "Fizz", "4", "Buzz", "Fizz", "7", "8", "Fizz", "Buzz", "11", "Fizz", "13", "14", "FizzBuzz"])])

## Problem 4 — doubling time

`years_to_double(rate)` returns how many **whole years** it takes for money to at least double at annual rate `rate` (compounded yearly). Use `while`.

In [ ]:
def years_to_double(rate):
    balance = 1.0
    years = 0
    while balance < 2.0:
        balance *= 1 + rate
        years += 1
    return years

In [ ]:
check(years_to_double, [((0.10,), 8), ((0.07,), 11), ((1.0,), 1), ((0.12,), 7)])

## Problem 5 — run-length compression

`compress("aaabccdddd")` → `"a3b1c2d4"`. Empty string → `""`.

*Hint for your plan:* keep track of the current character and how many times you've seen it in a row. What do you do when the character changes? What about the very last run?

In [ ]:
def compress(text):
    if not text:
        return ""
    result = ""
    current = text[0]
    count = 1
    for ch in text[1:]:
        if ch == current:
            count += 1
        else:
            result += f"{current}{count}"
            current = ch
            count = 1
    result += f"{current}{count}"      # flush the final run
    return result

In [ ]:
check(compress, [(("aaabccdddd",), "a3b1c2d4"), (("a",), "a1"), (("",), ""), (("abab",), "a1b1a1b1"), (("zzzzzz",), "z6")])

## Problem 6 — primes

`is_prime(n)` — only check divisors up to √n (i.e. while `d * d <= n`). Numbers below 2 are not prime.

In [ ]:
def is_prime(n):
    if n < 2:
        return False
    d = 2
    while d * d <= n:
        if n % d == 0:
            return False
        d += 1
    return True

In [ ]:
check(is_prime, [((2,), True), ((1,), False), ((17,), True), ((21,), False), ((97,), True), ((7919,), True), ((7917,), False)])

## Problem 7 — digit sum with a loop

`sum_digits(n)` for any non-negative integer, using `%` and `//` in a `while` loop (no strings).

In [ ]:
def sum_digits(n):
    total = 0
    while n > 0:
        total += n % 10
        n //= 10
    return total

In [ ]:
check(sum_digits, [((4827,), 21), ((0,), 0), ((1000001,), 2)])

## Problem 8 — first transaction over a limit

`first_over(amounts, limit)` returns the **index** of the first amount strictly above `limit`, or `-1` if there isn't one. Use `enumerate` and stop as soon as you find it.

In [ ]:
def first_over(amounts, limit):
    for i, amount in enumerate(amounts):
        if amount > limit:
            return i
    return -1

In [ ]:
check(first_over, [(([100, 500, 2000, 50], 400), 1), (([1, 2, 3], 10), -1), (([], 0), -1)])

## Problem 9 ★ — multiplication table

`times_table(n)` returns a single string: `n` lines, each with `n` numbers right-aligned in width 4 (`f"{x:>4}"`), lines joined by `"\n"`. Print the result to admire it.

In [ ]:
def times_table(n):
    lines = []
    for row in range(1, n + 1):
        line = ""
        for col in range(1, n + 1):
            line += f"{row * col:>4}"
        lines.append(line)
    return "\n".join(lines)

print(times_table(9))

In [ ]:
check(times_table, [((3,), "   1   2   3\n   2   4   6\n   3   6   9"), ((1,), "   1")])

## Problem 10 ★ — Collatz steps

Starting from `n`: if it's even, halve it; if it's odd, do 3n + 1. `collatz_steps(n)` returns how many steps it takes to reach 1.

In [ ]:
def collatz_steps(n):
    steps = 0
    while n != 1:
        n = n // 2 if n % 2 == 0 else 3 * n + 1
        steps += 1
    return steps

In [ ]:
check(collatz_steps, [((1,), 0), ((6,), 8), ((27,), 111)])